In [1]:
from pathlib import Path

import duckdb
import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / "data" / "master_listing.parquet"
OUT = ROOT / "data" / "feature_funnel.csv"

# Booleans and dates are kept apart from numbers: a flag or a date is not a
# measured quantity, and dates need converting before a model can use them.
KIND = {"BIGINT": "number", "DOUBLE": "number", "VARCHAR": "string",
        "BOOLEAN": "boolean", "DATE": "date", "TIMESTAMP": "date"}

step(f"Reading {MASTER.name}")
con = duckdb.connect()
src = f"read_parquet('{MASTER.as_posix()}')"
schema = con.execute(f"DESCRIBE SELECT * FROM {src}").fetchall()
n_rows = con.execute(f"SELECT count(*) FROM {src}").fetchone()[0]
funnel = pd.DataFrame({"variable": [c for c, *_ in schema], "dtype": [t for _, t, *_ in schema]})
done(f"{n_rows:,} rows x {len(funnel)} variables")

step("Extracting the source table from each variable name (split on '__')")
# The 4 join keys (unit_id, property_id, floor_plan_id, market) carry no prefix;
# they come from market.unit.
funnel["source_table"] = [v.split("__")[0] if "__" in v else "market_unit" for v in funnel["variable"]]
done(f"{funnel['source_table'].nunique()} source tables")
for table, n in funnel["source_table"].value_counts().items():
    note(f"{table:32s}{n:>4} variables")

step("Computing completeness and emptiness (% of rows non-null / null)")
counts = con.execute("SELECT " + ", ".join(f'count("{v}")' for v in funnel["variable"]) + f" FROM {src}").fetchone()
funnel["completeness_pct"] = [round(100 * c / n_rows, 2) for c in counts]
funnel["emptiness_pct"] = (100 - funnel["completeness_pct"]).round(2)
done(f"{(funnel['completeness_pct'] == 100).sum()} variables are 100% complete")
if (funnel["completeness_pct"] == 0).any():
    print(f"{RED}  ✗ fully empty: {funnel.loc[funnel['completeness_pct'] == 0, 'variable'].tolist()}{RESET}")

step("Classifying each variable as number or string")
funnel["type"] = funnel["dtype"].map(KIND).fillna("other")
for kind, g in funnel.groupby("type"):
    note(f"{kind:8s}{len(g):>4} variables, mean completeness {g['completeness_pct'].mean():6.2f}%")
if (funnel["type"] == "other").any():
    print(f"{RED}  ✗ unmapped dtypes: {sorted(funnel.loc[funnel['type'] == 'other', 'dtype'].unique())}{RESET}")

step("Funnel: variables kept at each completeness threshold")
for label, kept in [("all variables", len(funnel))] + [
        (f">= {t}% complete", (funnel["completeness_pct"] >= t).sum()) for t in (50, 80, 95, 100)]:
    color = GREEN if kept / len(funnel) >= 0.8 else YELLOW if kept / len(funnel) >= 0.5 else RED
    print(f"  {label:18s}{color}{'█' * round(40 * kept / len(funnel)):40s} {kept:>4}{RESET}")

step(f"Writing {OUT.relative_to(ROOT)}")
funnel = funnel[["variable", "source_table", "completeness_pct", "emptiness_pct", "type"]]
funnel.to_csv(OUT, index=False)
done(f"{len(funnel)} rows x {funnel.shape[1]} columns")
funnel


▶ Reading master_listing.parquet
  ✓ 967,661 rows x 378 variables

▶ Extracting the source table from each variable name (split on '__')
  ✓ 11 source tables
  • market_building                  270 variables
  • truth_building_params             23 variables
  • market_price_observation          20 variables
  • market_unit                       18 variables
  • pms_property                      15 variables
  • truth_lease_event                 13 variables
  • market_concession_event            9 variables
  • truth_concession                   5 variables
  • market_leasing_activity_daily      2 variables
  • truth_floorplan_market_rent        2 variables
  • market_floor_plan                  1 variables

▶ Computing completeness and emptiness (% of rows non-null / null)


  ✓ 71 variables are 100% complete
  ✗ fully empty: ['market_building__overall_quality_score', 'market_building__bedroom_quality_score', 'market_building__kitchen_quality_score', 'market_building__bathroom_quality_score', 'market_building__dining_room_quality_score', 'market_building__common_areas_quality_score', 'market_building__fitness_center_quality_score', 'market_building__laundry_room_quality_score', 'market_building__living_room_quality_score', 'market_building__main_entrance_quality_score', 'market_building__stairs_hallway_quality_score', 'market_building__swimming_pool_quality_score', 'market_building__walk_in_closet_quality_score']

▶ Classifying each variable as number or string
  • boolean  223 variables, mean completeness  81.09%
  • date      11 variables, mean completeness  83.31%
  • number   108 variables, mean completeness  66.03%
  • string    36 variables, mean completeness  78.00%

▶ Funnel: variables kept at each completeness threshold
  all variables     ███████

,variable,source_table,completeness_pct,emptiness_pct,type
0,unit_id,market_unit,100.00,0.00,string
1,property_id,market_unit,100.00,0.00,string
2,floor_plan_id,market_unit,100.00,0.00,string
3,market,market_unit,100.00,0.00,string
4,market_unit__unit_name,market_unit,89.22,10.78,string
...,...,...,...,...,...
373,pms_property__comps_disabled,pms_property,5.39,94.61,boolean
374,pms_property__pricing_disabled,pms_property,5.39,94.61,boolean
375,pms_property__fetch_pms_pricing_disabled,pms_property,5.39,94.61,boolean
376,pms_property__num_available_vacant_units,pms_property,5.39,94.61,number
